# 04 – LLM-Methode via Ollama

Ein lokales LLM bewertet für jede Tabelle + Textfenster, ob die Tabelleninformation
im Text wiederholt wird. Das LLM versteht semantische Äquivalenz:
`75%` = `three quarters` = `majority`.

**Vorteil gegenüber Regex/spaCy:**
- Erkennt inhaltliche Umformulierungen
- Kontextverständnis: unterscheidet echte Datenwerte von Zufallstreffern
- Gibt einen begründeten Score zurück

**Risiko:** LLM kann halluzinieren oder falsche Referenzen als korrekt bewerten.
Deshalb: strukturierter Prompt + Validierung gegen manuelle Annotation.

**Setup:**
```bash
# Ollama installieren: https://ollama.com
ollama pull llama3.2   # oder: mistral, phi3, gemma2
ollama serve           # Ollama-Server starten
pip install ollama
```

In [14]:
import json
import re
import pathlib
import pandas as pd
from tqdm.auto import tqdm

try:
    import ollama
    # Verbindung testen
    models = ollama.list()
    # Neue ollama-API (>=0.4) gibt Pydantic-Objekt zurück, keine dict
    if hasattr(models, 'models'):
        available = [m.model for m in models.models]
    else:
        available = [m['name'] for m in models.get('models', [])]
    print('Ollama erreichbar. Verfügbare Modelle:', available)
    OLLAMA_OK = True
except Exception as e:
    print(f'Ollama nicht verfügbar: {e}')
    print('Bitte Ollama installieren und starten: https://ollama.com')
    OLLAMA_OK = False

# Modell auswählen (erstes verfügbares, bevorzugt llama3.2)
PREFERRED = ['llama3.2', 'llama3', 'mistral', 'phi3', 'gemma2']
MODEL = None
if OLLAMA_OK:
    for pref in PREFERRED:
        if any(pref in a for a in available):
            MODEL = next(a for a in available if pref in a)
            break
    if not MODEL and available:
        MODEL = available[0]
    if not MODEL and available:
        MODEL = available[0]
    print(f'Verwendetes Modell: {MODEL}')

WINDOW_SIZE = 500

# Pfad-Erkennung
notebook_path = globals().get('__vsc_ipynb_file__')
NOTEBOOK_DIR = pathlib.Path(notebook_path).resolve().parent if notebook_path else pathlib.Path.cwd()
OUTPUT_DIR = NOTEBOOK_DIR / 'output'
if not OUTPUT_DIR.exists():
    OUTPUT_DIR = pathlib.Path.cwd() / 'output'
if not OUTPUT_DIR.exists():
    raise FileNotFoundError(f'output/ nicht gefunden in {NOTEBOOK_DIR}')

json_files = sorted(OUTPUT_DIR.glob('*.json'))
print(f'Gefundene Dateien: {len(json_files)}')

Ollama erreichbar. Verfügbare Modelle: ['llama3.2:latest', 'llama3.1:8b']
Verwendetes Modell: llama3.2:latest
Gefundene Dateien: 499


## Hilfsfunktionen

In [15]:
def get_text_window(fulltext, anchor, window=WINDOW_SIZE):
    search = anchor[:60].strip()
    if not search:
        return None
    idx = fulltext.find(search)
    if idx == -1:
        return None
    start = max(0, idx - window)
    end   = min(len(fulltext), idx + len(search) + window)
    return fulltext[start:end]


def parse_table_text(table, max_rows=8):
    """Tabelleninhalt als lesbaren String, auf max_rows Zeilen begrenzt."""
    try:
        raw = table.get('content', '{}')
        content = json.loads(raw) if isinstance(raw, str) else raw
        rows = content.get('data', [])[:max_rows]
        lines = [' | '.join(str(v)[:30] for v in row) for row in rows]
        return '\n'.join(lines)
    except Exception:
        return ''


PROMPT_TEMPLATE = """You are a scientific text analyst.

TABLE CAPTION: {caption}

TABLE CONTENT (first rows):
{table_content}

TEXT WINDOW (text surrounding the in-text reference to this table):
{text_window}

TASK: How much of the numerical/factual information from the table is also present
in the text window? Only count information that clearly refers to the same data,
not coincidental number matches.

Respond with ONLY a number between 0.0 and 1.0:
- 0.0 = no table information appears in the text
- 0.5 = roughly half the key values are mentioned in the text
- 1.0 = all table information is also present in the text

Score:"""


def llm_overlap_score(caption, table_content, text_window):
    """
    Fragt das LLM nach einem Overlap-Score (0.0–1.0).
    Gibt None zurück bei Fehler oder ungültiger Antwort.
    """
    if not OLLAMA_OK or not MODEL:
        return None
    prompt = PROMPT_TEMPLATE.format(
        caption=caption[:200],
        table_content=table_content[:800],
        text_window=text_window[:1000],
    )
    try:
        response = ollama.chat(
            model=MODEL,
            messages=[{'role': 'user', 'content': prompt}],
            options={'temperature': 0.0}  # deterministisch
        )
        answer = response['message']['content'].strip()
        # Ersten Float aus der Antwort extrahieren
        match = re.search(r'\b(0\.\d+|1\.0|0|1)\b', answer)
        if match:
            return round(float(match.group()), 4)
        return None
    except Exception as e:
        print(f'LLM-Fehler: {e}')
        return None


# Schnelltest
if OLLAMA_OK:
    test_score = llm_overlap_score(
        caption='Fatty acid composition of Antarctic fungi',
        table_content='Palmitic | 25.0% | Stearic | 9.4% | Oleic | 34.4%',
        text_window='The table shows palmitic (25%) and oleic fatty acids (34.4%) as main components.'
    )
    print(f'Testanfrage Score: {test_score}')
else:
    print('Ollama nicht verfügbar – Testanfrage übersprungen')

Testanfrage Score: 0.8


## Hauptanalyse

> ⚠️ Laufzeit: je nach Modell und Hardware ca. 1–5 Sekunden pro Tabelle.
> Bei 264 Tabellen ca. 5–20 Minuten. `MAX_TABLES` begrenzt die Analyse.

In [16]:
MAX_TABLES = None  # None = alle Tabellen; Zahl z.B. 100 für schnellen Test

rows = []
skipped_no_ollama = 0

for fpath in tqdm(json_files, desc='Verarbeite Dokumente'):
    with open(fpath, encoding='utf-8') as f:
        doc = json.load(f)

    year     = (doc.get('metadata') or {}).get('preprint_date', '')[:4]
    fulltext = doc.get('text', '')
    doi      = doc.get('doi', fpath.stem)
    if not year:
        continue

    for tab in doc.get('tables', []):
        if MAX_TABLES and len(rows) >= MAX_TABLES:
            break
        if not (tab.get('caption') or tab.get('name')):
            continue

        real_refs = [r for r in (tab.get('references') or []) if len(r) > 30]
        if not real_refs:
            continue

        table_content = parse_table_text(tab)
        if not table_content.strip():
            continue

        caption = tab.get('caption') or tab.get('name') or ''

        # Bestes Fenster nehmen (längste Referenz)
        best_ref = max(real_refs, key=len)
        window   = get_text_window(fulltext, best_ref, WINDOW_SIZE) or best_ref

        llm_score = llm_overlap_score(caption, table_content, window)
        if llm_score is None:
            skipped_no_ollama += 1
            # Trotzdem speichern mit None – für spätere manuelle Annotation

        rows.append({
            'doi'         : doi,
            'year'        : year,
            'table_name'  : tab.get('name', ''),
            'caption'     : caption[:80],
            'llm_score'   : llm_score,
            'text_window' : window[:300],   # für manuelle Kontrolle
            'table_sample': table_content[:200],
        })

    if MAX_TABLES and len(rows) >= MAX_TABLES:
        break

df = pd.DataFrame(rows)
print(f'Verarbeitete Tabellen : {len(df)}')
print(f'Ohne LLM-Score        : {skipped_no_ollama}')

df_scored = df.dropna(subset=['llm_score'])
if not df_scored.empty:
    print(f'Mit LLM-Score         : {len(df_scored)}')
    print(f'Ø LLM-Score           : {df_scored["llm_score"].mean():.4f}')
    print()
    print(df_scored.groupby('year')['llm_score'].mean().round(4))

Verarbeite Dokumente:   0%|          | 0/499 [00:00<?, ?it/s]

Verarbeitete Tabellen : 342
Ohne LLM-Score        : 0
Mit LLM-Score         : 342
Ø LLM-Score           : 0.6798

year
2022    0.8000
2023    0.5556
2024    0.6943
2025    0.6780
Name: llm_score, dtype: float64


## Vergleich aller drei Methoden

In [20]:
# Regex und spaCy Ergebnisse laden
results = {'llm': df_scored}

regex_csv = NOTEBOOK_DIR / 'regex_results.csv'
spacy_csv = NOTEBOOK_DIR / 'spacy_results.csv'

if regex_csv.exists():
    results['regex'] = pd.read_csv(regex_csv)
if spacy_csv.exists():
    df_spacy = pd.read_csv(spacy_csv)
    results['spacy'] = df_spacy

print('=== Methodenvergleich (Ø Overlap-Score) ===')
if 'regex' in results:
    print(f'  Regex        : {results["regex"]["overlap"].mean():.4f}')
if 'spacy' in results:
    # Spalte heißt je nach Ausführung entity_overlap oder combined
    spacy_col = next((c for c in ['combined', 'entity_overlap', 'ner_overlap'] 
                      if c in results['spacy'].columns), None)
    if spacy_col:
        print(f'  spaCy        : {results["spacy"][spacy_col].mean():.4f}')
    else:
        print('  spaCy        : CSV geladen aber keine Score-Spalte gefunden')
else:
    print('  spaCy        : spacy_results.csv nicht gefunden – zuerst 03_spacy.ipynb ausfuehren')
if not df_scored.empty:
    print(f'  LLM (Ollama) : {df_scored["llm_score"].mean():.4f}')

print()
print('Hinweis: Hoehere Score = mehr Tabelleninformation auch im Text.')
print('Zum Vergleich der Methoden: manuelle Annotation von ~50 Tabellen empfohlen.')

=== Methodenvergleich (Ø Overlap-Score) ===
  Regex        : 0.1237
  spaCy        : 0.0982
  LLM (Ollama) : 0.6798

Hinweis: Hoehere Score = mehr Tabelleninformation auch im Text.
Zum Vergleich der Methoden: manuelle Annotation von ~50 Tabellen empfohlen.


In [21]:
# Ergebnisse speichern
out_csv = NOTEBOOK_DIR / 'llm_results.csv'
df.to_csv(out_csv, index=False)
print(f'Gespeichert: {out_csv}')

# Beispiele ausgeben für manuelle Überprüfung
print()
print('=== Beispiele zur manuellen Kontrolle ===')
for _, row in df_scored.head(5).iterrows():
    print(f'Tabelle : {row["table_name"]} | Score: {row["llm_score"]}')
    print(f'Caption : {row["caption"][:70]}')
    print(f'Fenster : {row["text_window"][:150]}')
    print()

Gespeichert: C:\Users\Robin\TH_Koeln\Semester_6\DIS22\llm_results.csv

=== Beispiele zur manuellen Kontrolle ===
Tabelle : TABLE 1 | Score: 0.8
Caption : TABLE 1. Composition of the medium.
Fenster : ment  of  Environmental  Biology,  Sapienza 112 University of Rome, where they are conserved, respectively, with the numbers FBL 167, FBL 113 175 and 

Tabelle : TABLE 2 | Score: 0.8
Caption : TABLE 2. Substrates screening at a temperature of 25 °C
Fenster : In the present work, for all studied fungi we obtained high yields of mycelial dry matter in many of the substrates tested in preliminary screening (t

Tabelle : TABLE 3 | Score: 0.8
Caption : TABLE 3. Effect of temperature and medium on the yield of biomass
Fenster : d quality.

3.2 Fungal growth evaluation

The effects of several carbon and nitrogen source on biomass are shown in the table 2. For each fungus, the 

Tabelle : TABLE 5 | Score: 0.8
Caption : TABLE 5. Palmitic, stearic, oleic,  linoleic, linolenic acids composit
Fenster 